# Dynamic Agent Correctness Benchmark

Kaggle 提出向けの self-contained notebook です。

この notebook にはベンチマーク概要、ケース定義、ルールベース evaluator、検証と評価のデモを 1 本に集約しています。


## Structure

1. 共通 helper
2. 会議準備ケース
3. サンプル execution log
4. evaluator
5. validation summary
6. evaluation demo


In [ ]:
from __future__ import annotations

import json
from typing import Any


REQUIRED_TOP_LEVEL_FIELDS = [
    "task_id",
    "domain",
    "difficulty",
    "initial_request",
    "initial_state",
    "allowed_actions",
    "events",
    "goal_condition",
    "rubric",
]


def validate_case_structure(case: dict[str, Any]) -> list[str]:
    errors: list[str] = []
    for field_name in REQUIRED_TOP_LEVEL_FIELDS:
        if field_name not in case:
            errors.append(f"missing top-level field: {field_name}")

    initial_state = case.get("initial_state", {})
    for field_name in [
        "timezone",
        "deadline",
        "participants",
        "budget",
        "required_artifacts",
        "constraints",
    ]:
        if field_name not in initial_state:
            errors.append(f"missing initial_state field: {field_name}")

    rubric = case.get("rubric", {})
    for field_name in ["outcome", "process", "recovery"]:
        if field_name not in rubric:
            errors.append(f"missing rubric field: {field_name}")

    return errors


## Meeting Cases

会議準備ドメインの 5 ケースを notebook 内に埋め込みます。


In [ ]:
MEETING_CASES = [{'task_id': 'meeting_001',
  'domain': 'meeting_prep',
  'difficulty': 'easy',
  'initial_request': '来週の役員会議の準備をしてほしい。日程候補とアジェンダ案、事前説明資料の準備方針をまとめてください。',
  'initial_state': {'timezone': 'Asia/Tokyo',
                    'deadline': '2026-04-10T17:00:00+09:00',
                    'participants': ['A', 'B', 'C'],
                    'budget': 50000,
                    'required_artifacts': [{'artifact_id': 'agenda',
                                            'artifact_type': 'document',
                                            'required_fields': ['meeting_objective',
                                                                'agenda_items',
                                                                'owner',
                                                                'time_allocation']},
                                           {'artifact_id': 'briefing_doc',
                                            'artifact_type': 'document',
                                            'required_fields': ['summary',
                                                                'decision_points',
                                                                'risks',
                                                                'next_actions']}],
                    'constraints': ['参加者全員が参加可能な時間帯を候補に含める', '議題は3件以上必要', '事前説明資料は会議前日までに完成させる'],
                    'task_dependencies': [{'before': 'collect_availability',
                                           'after': 'propose_schedule',
                                           'reason': '参加可能時間を確認してから候補日程を作る'},
                                          {'before': 'define_agenda',
                                           'after': 'write_briefing_doc',
                                           'reason': '議題を固めてから説明資料を書く'}],
                    'reference_data': {'meeting_type': 'executive', 'agenda_min_items': 3}},
  'allowed_actions': ['ask_clarification',
                      'propose_plan',
                      'update_plan',
                      'create_artifact',
                      'revise_artifact',
                      'confirm_state',
                      'finalize'],
  'events': [{'turn': 3,
              'type': 'state_change',
              'message': '役員Dの参加が必須になりました。締切は明日正午です。',
              'delta': {'participants_added': ['D'], 'deadline': '2026-04-09T12:00:00+09:00'},
              'expected_artifact_updates': ['agenda', 'briefing_doc'],
              'expected_replan_within_turns': 2}],
  'goal_condition': {'must_satisfy_latest_state': True,
                     'required_artifacts': ['agenda', 'briefing_doc'],
                     'no_constraint_violation': True,
                     'must_acknowledge_changes': True},
  'rubric': {'outcome': ['最終成果物が参加者Dを反映している',
                         '更新後の締切に整合している',
                         'agenda と briefing_doc の必須項目が埋まっている'],
             'process': ['変更後に旧日程案を前提とせず再評価した', '資料更新の優先度を引き上げた'],
             'recovery': ['2ターン以内に再計画を表明した', '影響を受ける成果物を特定した']},
  'notes': '基本ケース。参加者追加と締切前倒しに対する標準的な再計画能力を評価する。'},
 {'task_id': 'meeting_002',
  'domain': 'meeting_prep',
  'difficulty': 'easy',
  'initial_request': '新規事業会議の準備をしてください。会議の候補日程、アジェンダ、参加者向けの1ページ要約を用意したいです。',
  'initial_state': {'timezone': 'Asia/Tokyo',
                    'deadline': '2026-04-15T18:00:00+09:00',
                    'participants': ['PM', 'Finance', 'Legal'],
                    'budget': 30000,
                    'required_artifacts': [{'artifact_id': 'agenda',
                                            'artifact_type': 'document',
                                            'required_fields': ['purpose',
                                                                'agenda_items',
                                                                'decision_needed']},
                                           {'artifact_id': 'one_pager',
                                            'artifact_type': 'document',
                                            'required_fields': ['background',
                                                                'proposal',
                                                                'open_questions']}],
                    'constraints': ['法務レビュー後でないと配布資料を確定できない', '予算3万円以内で会議運営する'],
                    'task_dependencies': [{'before': 'legal_review',
                                           'after': 'finalize_one_pager',
                                           'reason': '法務レビュー後に要約資料を確定する必要がある'}],
                    'reference_data': {'room_type': 'hybrid'}},
  'allowed_actions': ['ask_clarification',
                      'propose_plan',
                      'update_plan',
                      'create_artifact',
                      'revise_artifact',
                      'confirm_state',
                      'finalize'],
  'events': [{'turn': 2,
              'type': 'new_constraint',
              'message': '会場費が上がるので、会議運営予算は2万円までに変更してください。',
              'delta': {'budget': 20000},
              'expected_artifact_updates': ['agenda', 'one_pager'],
              'expected_replan_within_turns': 1}],
  'goal_condition': {'must_satisfy_latest_state': True,
                     'required_artifacts': ['agenda', 'one_pager'],
                     'no_constraint_violation': True,
                     'must_acknowledge_changes': True},
  'rubric': {'outcome': ['最新予算制約の範囲で計画されている', 'one_pager が法務レビュー前提を壊していない'],
             'process': ['予算変更を受けてコスト前提を更新した', '依存関係 legal_review -> finalize_one_pager を維持した'],
             'recovery': ['1ターン以内に予算変更への対応を表明した']},
  'notes': '予算制約と依存関係維持を確認するケース。'},
 {'task_id': 'meeting_003',
  'domain': 'meeting_prep',
  'difficulty': 'medium',
  'initial_request': '四半期レビュー会議の準備をお願いします。参加者調整、アジェンダ案、レビュー資料の完成まで段取りしたいです。',
  'initial_state': {'timezone': 'Asia/Tokyo',
                    'deadline': '2026-04-20T10:00:00+09:00',
                    'participants': ['CEO', 'COO', 'CFO', 'Sales'],
                    'budget': 80000,
                    'required_artifacts': [{'artifact_id': 'agenda',
                                            'artifact_type': 'document',
                                            'required_fields': ['agenda_items',
                                                                'decision_points',
                                                                'timebox']},
                                           {'artifact_id': 'review_deck',
                                            'artifact_type': 'slides',
                                            'required_fields': ['revenue_summary',
                                                                'forecast',
                                                                'issues',
                                                                'asks']}],
                    'constraints': ['レビュー資料には最新売上数値を使う', '会議は90分以内', '参加者全員の出席を優先する'],
                    'task_dependencies': [{'before': 'fetch_latest_sales',
                                           'after': 'draft_review_deck',
                                           'reason': '売上数値取得後に資料を起こす必要がある'},
                                          {'before': 'draft_review_deck',
                                           'after': 'final_review_deck',
                                           'reason': 'ドラフトなしに最終化できない'}],
                    'reference_data': {'deck_version': 'v1', 'sales_snapshot_date': '2026-04-18'}},
  'allowed_actions': ['ask_clarification',
                      'propose_plan',
                      'update_plan',
                      'create_artifact',
                      'revise_artifact',
                      'confirm_state',
                      'finalize'],
  'events': [{'turn': 2,
              'type': 'artifact_update',
              'message': '売上データの最新版が届きました。以前の数字は古いので差し替えが必要です。',
              'delta': {'reference_data': {'deck_version': 'v2',
                                           'sales_snapshot_date': '2026-04-19'}},
              'expected_artifact_updates': ['review_deck'],
              'expected_replan_within_turns': 1},
             {'turn': 4,
              'type': 'stakeholder_change',
              'message': 'CFOは会議冒頭30分しか参加できません。意思決定が必要な議題を前半に寄せてください。',
              'delta': {'participant_availability_note': 'CFO available first 30 minutes only'},
              'expected_artifact_updates': ['agenda'],
              'expected_replan_within_turns': 1}],
  'goal_condition': {'must_satisfy_latest_state': True,
                     'required_artifacts': ['agenda', 'review_deck'],
                     'no_constraint_violation': True,
                     'must_acknowledge_changes': True},
  'rubric': {'outcome': ['review_deck が最新売上数値に基づいている', 'アジェンダ前半に意思決定項目が配置されている'],
             'process': ['v1 データを使い続けず v2 へ更新した', 'CFO の制約を受けて議題順を調整した'],
             'recovery': ['2 回のイベントに対してそれぞれ即時に再計画した']},
  'notes': 'データ更新と参加制約変更が連続する中難度ケース。'},
 {'task_id': 'meeting_004',
  'domain': 'meeting_prep',
  'difficulty': 'medium',
  'initial_request': '採用方針会議の準備を進めてください。候補日程、面接枠の論点整理、会議用メモを作りたいです。',
  'initial_state': {'timezone': 'Asia/Tokyo',
                    'deadline': '2026-04-22T15:00:00+09:00',
                    'participants': ['HR', 'HiringManager', 'Ops'],
                    'budget': 20000,
                    'required_artifacts': [{'artifact_id': 'agenda',
                                            'artifact_type': 'document',
                                            'required_fields': ['objective', 'topics', 'owner']},
                                           {'artifact_id': 'meeting_memo',
                                            'artifact_type': 'document',
                                            'required_fields': ['hiring_plan',
                                                                'interview_capacity',
                                                                'risks']}],
                    'constraints': ['Ops の承認なしに面接枠は確定できない', 'meeting_memo には最新採用人数見込みを反映する'],
                    'task_dependencies': [{'before': 'collect_hiring_forecast',
                                           'after': 'draft_meeting_memo',
                                           'reason': '採用人数見込みがないとメモを起こせない'},
                                          {'before': 'ops_approval',
                                           'after': 'finalize_interview_capacity',
                                           'reason': '承認前に枠を確定してはいけない'}],
                    'reference_data': {'forecast_hires': 6}},
  'allowed_actions': ['ask_clarification',
                      'propose_plan',
                      'update_plan',
                      'create_artifact',
                      'revise_artifact',
                      'confirm_state',
                      'finalize'],
  'events': [{'turn': 3,
              'type': 'state_change',
              'message': '採用人数見込みは6名から9名に更新されました。',
              'delta': {'reference_data': {'forecast_hires': 9}},
              'expected_artifact_updates': ['meeting_memo'],
              'expected_replan_within_turns': 1},
             {'turn': 5,
              'type': 'stakeholder_change',
              'message': 'Ops が本日不在のため承認は明日になります。承認前提での確定記述は避けてください。',
              'delta': {'ops_approval_available': False},
              'expected_artifact_updates': ['agenda', 'meeting_memo'],
              'expected_replan_within_turns': 1}],
  'goal_condition': {'must_satisfy_latest_state': True,
                     'required_artifacts': ['agenda', 'meeting_memo'],
                     'no_constraint_violation': True,
                     'must_acknowledge_changes': True},
  'rubric': {'outcome': ['meeting_memo が最新採用人数見込み9名を反映している', 'Ops 承認前に確定表現をしていない'],
             'process': ['採用人数更新をメモに反映した', '依存関係 ops_approval -> finalize_interview_capacity を守った'],
             'recovery': ['承認遅延を受けて成果物の確定範囲を調整した']},
  'notes': '依存関係を壊しやすいケース。unsafe_commit を見つけやすい。'},
 {'task_id': 'meeting_005',
  'domain': 'meeting_prep',
  'difficulty': 'medium',
  'initial_request': '社外パートナーとの定例会議の準備をしてほしいです。日程調整、配布アジェンダ、共有メモの準備をお願いします。',
  'initial_state': {'timezone': 'Asia/Tokyo',
                    'deadline': '2026-04-25T16:00:00+09:00',
                    'participants': ['InternalPM', 'PartnerLead', 'Engineering'],
                    'budget': 10000,
                    'required_artifacts': [{'artifact_id': 'agenda',
                                            'artifact_type': 'document',
                                            'required_fields': ['topics',
                                                                'owners',
                                                                'external_share_ok']},
                                           {'artifact_id': 'shared_memo',
                                            'artifact_type': 'document',
                                            'required_fields': ['context',
                                                                'discussion_points',
                                                                'followups']}],
                    'constraints': ['社外共有資料には未公開ロードマップを含めない', 'パートナー向け資料は対外公開可否を確認してから送付する'],
                    'task_dependencies': [{'before': 'check_external_share_policy',
                                           'after': 'send_agenda',
                                           'reason': '共有可否確認前に送付してはいけない'}],
                    'reference_data': {'share_policy': 'internal draft not approved'}},
  'allowed_actions': ['ask_clarification',
                      'propose_plan',
                      'update_plan',
                      'create_artifact',
                      'revise_artifact',
                      'confirm_state',
                      'finalize'],
  'events': [{'turn': 2,
              'type': 'artifact_update',
              'message': '共有候補メモに未公開ロードマップが含まれていると判明しました。社外共有版を分離してください。',
              'delta': {'reference_data': {'share_policy': 'split internal and external versions'}},
              'expected_artifact_updates': ['agenda', 'shared_memo'],
              'expected_replan_within_turns': 1},
             {'turn': 4,
              'type': 'new_constraint',
              'message': '先方の都合で会議時間は45分に短縮してください。',
              'delta': {'meeting_duration_limit_minutes': 45},
              'expected_artifact_updates': ['agenda'],
              'expected_replan_within_turns': 1}],
  'goal_condition': {'must_satisfy_latest_state': True,
                     'required_artifacts': ['agenda', 'shared_memo'],
                     'no_constraint_violation': True,
                     'must_acknowledge_changes': True},
  'rubric': {'outcome': ['社外共有物に未公開情報が残っていない', '45分制約に合わせてアジェンダが再編されている'],
             'process': ['社内版と社外版の分離を明示した', '共有可否確認の依存関係を守った'],
             'recovery': ['2つのイベント後にそれぞれ不要な情報と時間超過を解消した']},
  'notes': '情報ガバナンスと時間制約の両方を扱うケース。'}]

print(f"Loaded {len(MEETING_CASES)} benchmark cases")
print([case["task_id"] for case in MEETING_CASES])


## Sample Execution Log

評価デモ用の execution log です。


In [ ]:
SAMPLE_EXECUTION_LOGS = {'meeting_001': {'task_id': 'meeting_001',
                 'actions': [{'turn': 1,
                              'action_type': 'propose_plan',
                              'acknowledged_event_turns': [],
                              'artifact_updates': [],
                              'notes': '参加可能時間確認、議題整理、資料作成の順で進める'},
                             {'turn': 3,
                              'action_type': 'confirm_state',
                              'acknowledged_event_turns': [3],
                              'artifact_updates': [],
                              'notes': '参加者D追加と締切前倒しを確認'},
                             {'turn': 4,
                              'action_type': 'update_plan',
                              'acknowledged_event_turns': [3],
                              'artifact_updates': ['agenda', 'briefing_doc'],
                              'notes': '旧日程案を破棄し、資料修正を優先する'},
                             {'turn': 5,
                              'action_type': 'finalize',
                              'acknowledged_event_turns': [],
                              'artifact_updates': ['agenda', 'briefing_doc'],
                              'notes': '最新状態に基づいて成果物を確定'}],
                 'completed_tasks': ['collect_availability',
                                     'propose_schedule',
                                     'define_agenda',
                                     'write_briefing_doc'],
                 'constraint_violations': [],
                 'unsafe_commit': False,
                 'final_state': {'deadline': '2026-04-09T12:00:00+09:00',
                                 'participants': ['A', 'B', 'C', 'D'],
                                 'budget': 50000,
                                 'reference_data': {'meeting_type': 'executive',
                                                    'agenda_min_items': 3}},
                 'final_artifacts': {'agenda': {'fields_completed': ['meeting_objective',
                                                                     'agenda_items',
                                                                     'owner',
                                                                     'time_allocation']},
                                     'briefing_doc': {'fields_completed': ['summary',
                                                                           'decision_points',
                                                                           'risks',
                                                                           'next_actions']}}}}

print("Available sample execution logs:", list(SAMPLE_EXECUTION_LOGS.keys()))


## Rule-Based Evaluator

最終成果物、プロセス、復帰力を採点します。


In [ ]:
# ruff: noqa: I001

"""Rule-based evaluator for Dynamic Agent Correctness Benchmark."""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any


FAILURE_LABELS = {
    "state_staleness",
    "missing_replan",
    "partial_replan",
    "invalid_dependency",
    "goal_drift",
    "unsafe_commit",
    "constraint_violation",
    "artifact_inconsistency",
}


@dataclass
class EvaluationResult:
    outcome_score: int
    process_score: int
    recovery_score: int
    total_score: int
    failure_labels: list[str] = field(default_factory=list)
    deductions: list[str] = field(default_factory=list)


def _safe_list(value: Any) -> list[Any]:
    return value if isinstance(value, list) else []


def _artifact_map(final_artifacts: dict[str, Any]) -> dict[str, dict[str, Any]]:
    return {
        artifact_id: artifact_value
        for artifact_id, artifact_value in final_artifacts.items()
        if isinstance(artifact_value, dict)
    }


def _merge_state_delta(expected_state: dict[str, Any], delta: dict[str, Any]) -> None:
    for key, value in delta.items():
        if key == "participants_added":
            participants = _safe_list(expected_state.get("participants"))
            expected_state["participants"] = participants + [
                participant for participant in _safe_list(value) if participant not in participants
            ]
            continue

        if isinstance(value, dict):
            current_value = expected_state.get(key, {})
            if not isinstance(current_value, dict):
                current_value = {}
            merged_value = dict(current_value)
            _merge_state_delta(merged_value, value)
            expected_state[key] = merged_value
            continue

        expected_state[key] = value


def _compare_expected_state(
    expected_state: dict[str, Any],
    actual_state: dict[str, Any],
    deductions: list[str],
    failure_labels: set[str],
    path_prefix: str = "",
) -> int:
    penalty = 0
    for key, expected_value in expected_state.items():
        if key not in actual_state:
            penalty += 10
            deductions.append(f"final state missing key: {path_prefix}{key} (-10)")
            failure_labels.add("state_staleness")
            continue

        actual_value = actual_state[key]
        if isinstance(expected_value, dict):
            if not isinstance(actual_value, dict):
                penalty += 10
                deductions.append(f"final state type mismatch for {path_prefix}{key} (-10)")
                failure_labels.add("state_staleness")
                continue
            penalty += _compare_expected_state(
                expected_value,
                actual_value,
                deductions,
                failure_labels,
                path_prefix=f"{path_prefix}{key}.",
            )
            continue

        if isinstance(expected_value, list):
            if actual_value != expected_value:
                penalty += 10
                deductions.append(f"final state mismatch for {path_prefix}{key} (-10)")
                failure_labels.add("state_staleness")
            continue

        if actual_value != expected_value:
            penalty += 10
            deductions.append(f"final state mismatch for {path_prefix}{key} (-10)")
            failure_labels.add("state_staleness")

    return penalty


def _task_order_map(completed_tasks: list[Any]) -> dict[str, int]:
    order_map: dict[str, int] = {}
    for index, task_name in enumerate(completed_tasks):
        if isinstance(task_name, str) and task_name not in order_map:
            order_map[task_name] = index
    return order_map


def _build_tracked_expected_state(initial_state: dict[str, Any], events: list[Any]) -> dict[str, Any]:
    tracked_state: dict[str, Any] = {}

    for key in ["deadline", "participants", "budget", "reference_data"]:
        if key in initial_state:
            value = initial_state[key]
            tracked_state[key] = dict(value) if isinstance(value, dict) else value

    for event in events:
        delta = event.get("delta", {})
        if isinstance(delta, dict):
            _merge_state_delta(tracked_state, delta)

    return tracked_state


def evaluate_case(case: dict[str, Any], execution_log: dict[str, Any]) -> EvaluationResult:
    outcome_score = 100
    process_score = 100
    recovery_score = 100
    deductions: list[str] = []
    failure_labels: set[str] = set()

    initial_state = case.get("initial_state", {})
    required_artifacts = initial_state.get("required_artifacts", [])
    dependencies = _safe_list(initial_state.get("task_dependencies"))
    events = _safe_list(case.get("events"))
    goal_condition = case.get("goal_condition", {})

    actions = _safe_list(execution_log.get("actions"))
    final_state = execution_log.get("final_state", {})
    final_artifacts = _artifact_map(execution_log.get("final_artifacts", {}))
    completed_tasks = _safe_list(execution_log.get("completed_tasks"))
    task_order = _task_order_map(completed_tasks)

    required_artifact_ids = {artifact["artifact_id"] for artifact in required_artifacts}

    for artifact in required_artifacts:
        artifact_id = artifact["artifact_id"]
        if artifact_id not in final_artifacts:
            outcome_score -= 25
            deductions.append(f"missing required artifact: {artifact_id} (-25)")
            failure_labels.add("goal_drift")
            continue

        actual_artifact = final_artifacts[artifact_id]
        completed_fields = set(_safe_list(actual_artifact.get("fields_completed")))
        for field_name in _safe_list(artifact.get("required_fields")):
            if field_name not in completed_fields:
                outcome_score -= 5
                deductions.append(f"artifact {artifact_id} missing field: {field_name} (-5)")
                failure_labels.add("artifact_inconsistency")

    if goal_condition.get("must_satisfy_latest_state"):
        expected_state = _build_tracked_expected_state(initial_state, events)
        outcome_score -= _compare_expected_state(
            expected_state,
            final_state,
            deductions,
            failure_labels,
        )

    if goal_condition.get("no_constraint_violation"):
        if _safe_list(execution_log.get("constraint_violations")):
            for violation in _safe_list(execution_log.get("constraint_violations")):
                outcome_score -= 20
                deductions.append(f"constraint violation: {violation} (-20)")
                failure_labels.add("constraint_violation")

    for dependency in dependencies:
        after_task = dependency.get("after")
        before_task = dependency.get("before")
        if after_task not in task_order:
            continue
        if before_task not in task_order:
            process_score -= 20
            deductions.append(
                f"dependency violated: {before_task} must happen before {after_task} (-20)"
            )
            failure_labels.add("invalid_dependency")
            continue
        if task_order[before_task] > task_order[after_task]:
            process_score -= 20
            deductions.append(
                f"dependency order violated: {before_task} after {after_task} (-20)"
            )
            failure_labels.add("invalid_dependency")

    action_turn_map: dict[int, list[dict[str, Any]]] = {}
    for action in actions:
        turn = action.get("turn")
        if isinstance(turn, int):
            action_turn_map.setdefault(turn, []).append(action)

    for event in events:
        turn = event.get("turn")
        expected_replan = event.get("expected_replan_within_turns", 0)
        responded = False
        updated_artifacts = False
        acknowledged = False
        expected_artifact_updates = set(_safe_list(event.get("expected_artifact_updates")))
        for action in actions:
            action_turn = action.get("turn")
            if not isinstance(action_turn, int) or not isinstance(turn, int):
                continue
            if action_turn < turn:
                continue
            if action_turn > turn + expected_replan:
                continue
            if action.get("action_type") in {"update_plan", "confirm_state"}:
                responded = True
            if turn in _safe_list(action.get("acknowledged_event_turns")):
                acknowledged = True
            artifact_updates = set(_safe_list(action.get("artifact_updates")))
            if expected_artifact_updates:
                if expected_artifact_updates.issubset(artifact_updates):
                    updated_artifacts = True
            elif artifact_updates and artifact_updates.issubset(required_artifact_ids):
                updated_artifacts = True

        if not responded:
            recovery_score -= 15
            process_score -= 15
            deductions.append(
                f"no timely replan after event turn {turn} (-15 recovery, -15 process)"
            )
            failure_labels.add("missing_replan")
        if not acknowledged and goal_condition.get("must_acknowledge_changes", True):
            recovery_score -= 10
            deductions.append(f"event turn {turn} was not explicitly acknowledged (-10)")
            failure_labels.add("state_staleness")
        if not updated_artifacts:
            recovery_score -= 10
            deductions.append(f"no artifact update observed after event turn {turn} (-10)")
            failure_labels.add("partial_replan")

    if execution_log.get("unsafe_commit"):
        process_score -= 20
        deductions.append("unsafe commit detected (-20)")
        failure_labels.add("unsafe_commit")

    delivered_artifacts = set(final_artifacts.keys())
    if required_artifact_ids and not required_artifact_ids.issubset(delivered_artifacts):
        failure_labels.add("goal_drift")

    outcome_score = max(0, outcome_score)
    process_score = max(0, process_score)
    recovery_score = max(0, recovery_score)
    total_score = round(outcome_score * 0.40 + process_score * 0.35 + recovery_score * 0.25)

    return EvaluationResult(
        outcome_score=outcome_score,
        process_score=process_score,
        recovery_score=recovery_score,
        total_score=total_score,
        failure_labels=sorted(label for label in failure_labels if label in FAILURE_LABELS),
        deductions=deductions,
    )


## Validation Summary

ケース構造の妥当性を確認します。


In [ ]:
validation_summary = []
for case in MEETING_CASES:
    errors = validate_case_structure(case)
    validation_summary.append(
        {
            "task_id": case["task_id"],
            "status": "OK" if not errors else "ERROR",
            "errors": errors,
            "events": len(case["events"]),
            "participants": len(case["initial_state"]["participants"]),
            "artifacts": len(case["initial_state"]["required_artifacts"]),
        }
    )

print(json.dumps(validation_summary, ensure_ascii=False, indent=2))


## Evaluation Demo

`meeting_001` をサンプル log で評価します。


In [ ]:
demo_case = next(
    case for case in MEETING_CASES if case["task_id"] == "meeting_001"
)
demo_execution_log = SAMPLE_EXECUTION_LOGS["meeting_001"]
demo_result = evaluate_case(demo_case, demo_execution_log)

evaluation_payload = {
    "task_id": demo_case["task_id"],
    "outcome_score": demo_result.outcome_score,
    "process_score": demo_result.process_score,
    "recovery_score": demo_result.recovery_score,
    "total_score": demo_result.total_score,
    "failure_labels": demo_result.failure_labels,
    "deductions": demo_result.deductions,
}

print(json.dumps(evaluation_payload, ensure_ascii=False, indent=2))
